In [ ]:
# !pip install pandas numpy scikit-learn tensorflow matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
df = pd.read_csv("dataset/cropdata_updated.csv")
df.head()

In [ ]:
# Rename columns if needed
df.columns = [col.lower() for col in df.columns]

# Example mapping (adjust based on dataset)
df = df.rename(columns={
    "temperature": "temp",
    "humidity": "humidity",
    "result": "irrigation"
})

# Ensure irrigation column has only 0 and 1
df["irrigation"] = df["irrigation"].astype(int)
df["irrigation"] = (df["irrigation"] > 0).astype(int)  # Convert to binary if needed

# Add crop column (simulate)
crop_map = {"rice":0, "wheat":1, "maize":2, "vegetables":3, "pulses":4}
df["crop"] = np.random.randint(0,5,len(df))

# Generate synthetic raw soil (0?1023)
df["soil_raw"] = (
    1023 * (1 - (
        0.5 * (df["humidity"] / 100) +
        0.3 * (1 / (df["temp"] + 1))
    ))
)

# Add noise
df["soil_raw"] += np.random.randint(-50, 50, len(df))

# Clip values to sensor range
df["soil_raw"] = df["soil_raw"].clip(0, 1023)

df["soil"] = 1 - (df["soil_raw"] / 1023)

X = df[["temp","humidity","crop","soil"]]
y = df["irrigation"]

print(f"Unique values in irrigation column: {df['irrigation'].unique()}")

In [ ]:
print(X.head())

In [ ]:
scaler = MinMaxScaler()

X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
from tensorflow.keras.layers import Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import L2

# Improved model architecture with BatchNormalization and L2 regularization
model = Sequential([
    Input(shape=(4,)),
    Dense(64, activation='relu', kernel_regularizer=L2(0.001)),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(32, activation='relu', kernel_regularizer=L2(0.001)),
    BatchNormalization(),
    Dropout(0.25),
    
    Dense(16, activation='relu', kernel_regularizer=L2(0.001)),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(8, activation='relu', kernel_regularizer=L2(0.001)),
    Dropout(0.1),
    
    Dense(1, activation='sigmoid')
])

# Use custom learning rate for better convergence
optimizer = Adam(learning_rate=0.0005)  # Reduced learning rate
model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])

print(model.summary())

In [ ]:
# Early stopping and learning rate reduction
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=0.00001,
    verbose=1
)

# Calculate class weights for imbalanced data
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

print(f"Class weights: {class_weight_dict}")

history = model.fit(
    X_train, y_train,
    epochs=300,  # Increased epochs
    batch_size=8,  # Smaller batch size for better convergence
    validation_data=(X_test, y_test),
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weight_dict,  # Handle class imbalance
    verbose=1
)

In [ ]:
# Accuracy plot
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train","Validation"])
plt.show()

# Loss plot
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(["Train","Validation"])
plt.show()

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print("Final Accuracy:", acc)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Detailed metrics
print("\n" + "="*50)
print("DETAILED MODEL EVALUATION")
print("="*50)
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=['No Irrigation', 'Irrigation Needed']))

# Plot confusion matrix
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
print("\n" + "="*60)
print("DIAGNOSIS: CHECKING FOR LABEL ISSUES")
print("="*60)
print(f"\nUnique values in y_test: {np.unique(y_test)}")
print(f"Unique values in y_pred: {np.unique(y_pred)}")
print(f"y_test value counts:\n{pd.Series(y_test).value_counts()}")
print(f"y_pred value counts:\n{pd.Series(y_pred.flatten()).value_counts()}")

In [ ]:
sample = np.array([[30, 60, 0.5, 0]])  # temp, humidity, soil, crop
sample_scaled = scaler.transform(sample)

prediction = model.predict(sample_scaled)
print("Irrigation Needed:", int(prediction[0][0] > 0.5))

In [ ]:
# Save model in modern .keras format (replaces deprecated .h5)
model.save("irrigation_model.keras")
print("? Model saved as irrigation_model.keras")

import pickle
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("? Scaler saved as scaler.pkl")